In [1]:
##%pip install statsmodels
##%pip install scikit-learn
##%pip install matplotlib
##%pip install pmdarima

In [2]:
import sys
sys.path.append("C:/Users/Powan/Desktop/Maestria/TFM/edev_models/modelos")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt



from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

import pmdarima as pm
from collections import Counter

In [3]:

from construir_dataset_maestro import (
        construir_espina_horaria, construir_dataset_diario, dividir_train_val_test,
    )

espina = construir_espina_horaria()          # para EDA/correlaciones
dataset = construir_dataset_diario()          # para modelado
train, val, test = dividir_train_val_test(dataset)

C:\Users/Powan/Desktop/Maestria/TFM/edev_models/modelos\construir_dataset_maestro.py:149: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=params)
C:\Users/Powan/Desktop/Maestria/TFM/edev_models/modelos\construir_dataset_maestro.py:149: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=params)
C:\Users/Powan/Desktop/Maestria/TFM/edev_models/modelos\construir_dataset_maestro.py:149: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=

UndefinedColumn: column "date" does not exist
LINE 1: SELECT * FROM esios_capacity_available WHERE date BETWEEN '2...
                                                     ^


In [ ]:
print(dataset.columns.tolist())

# Preparacion de datos
Para esta idea inicial el target va a ser la media de todas las horas.
Para esto se va a crear una columna nueva, y se van a elimnar las horas del dataset a utilizar

In [ ]:
def preparar_chunk_train_test(
    dataset: pd.DataFrame,
    fecha_inicio: str = None,
    fecha_fin: str = None,
    test_size: float = 0.2,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Selects a date-range chunk of `dataset` and splits it chronologically:
    train = earliest (1 - test_size) of the chunk, test = the most recent
    test_size fraction. Never shuffles.

    fecha_inicio / fecha_fin: 'YYYY-MM-DD' strings, or None to use the
    full available range on that side.
    """
    df = dataset.copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    if fecha_inicio is not None:
        df = df[df.index >= pd.Timestamp(fecha_inicio)]
    if fecha_fin is not None:
        df = df[df.index <= pd.Timestamp(fecha_fin)]

    n_test = int(len(df) * test_size)
    if n_test == 0 or n_test == len(df):
        raise ValueError(f"test_size={test_size} leaves an empty split for {len(df)} rows")

    train = df.iloc[:-n_test]
    test = df.iloc[-n_test:]
    return train, test

In [ ]:
# 1. Pick your chunk -- adjust dates or test_size as needed
train_raw, test_raw = preparar_chunk_train_test(
    dataset,
    fecha_inicio="2023-01-01",
    fecha_fin="2026-08-14",
    test_size=0.2,
)

# 2. Build target, drop the leak columns, from each split independently
def preparar_xy(df):
    price_cols = [c for c in df.columns if c.startswith("price_h")]
    y = df[price_cols].mean(axis=1).rename("price_mean_d1")
    feature_cols = [c for c in df.columns if c not in price_cols]
    return df[feature_cols], y

X_train, y_train = preparar_xy(train_raw)
X_test, y_test = preparar_xy(test_raw)

In [ ]:
X_test.head(5)

# Feature selection
Spearman

In [ ]:
from scipy.stats import spearmanr

In [ ]:
def select_features_spearman(
    X: pd.DataFrame,
    y: pd.Series,
    target_threshold: float = 0.10,
    collinearity_threshold: float = 0.90,
    p_value_max: float = 0.05,
    min_overlap: int = 30,
) -> dict:
    """
    Fit this on the TRAIN split only -- selection must not see val/test rows.

    Stage 1: keep features whose Spearman correlation with the target is
    both meaningful (|rho| >= target_threshold) and significant (p < p_value_max).
    Stage 2: among survivors, greedily drop redundant features -- for any pair
    with |pairwise rho| >= collinearity_threshold, keep the one more correlated
    with the target and drop the other.

    Returns:
      selected           -- final feature list, ordered by |rho| with target
      target_corr        -- rho with target for every numeric input feature
      dropped_low_corr    -- cut in stage 1 (weak or non-significant)
      dropped_collinear   -- cut in stage 2, redundant with a stronger feature
    """
    X_num = X.select_dtypes(include=[np.number])

    rho, pval = {}, {}
    for col in X_num.columns:
        mask = X_num[col].notna() & y.notna()
        if mask.sum() < min_overlap:
            rho[col], pval[col] = np.nan, np.nan
            continue
        r, p = spearmanr(X_num.loc[mask, col], y[mask])
        rho[col], pval[col] = r, p

    target_corr = pd.Series(rho)
    pvals = pd.Series(pval)

    survivors = [
        c for c in X_num.columns
        if pd.notna(target_corr[c])
        and abs(target_corr[c]) >= target_threshold
        and pvals[c] < p_value_max
    ]
    dropped_low_corr = [c for c in X_num.columns if c not in survivors]

    corr_matrix = X_num[survivors].corr(method="spearman").abs()
    ordered = target_corr[survivors].abs().sort_values(ascending=False).index.tolist()

    selected, dropped_collinear, excluded = [], [], set()
    for col in ordered:
        if col in excluded:
            continue
        selected.append(col)
        for other in ordered:
            if other == col or other in excluded or other in selected:
                continue
            if corr_matrix.loc[col, other] >= collinearity_threshold:
                excluded.add(other)
                dropped_collinear.append(other)

    return {
        "selected": selected,
        "target_corr": target_corr.sort_values(key=abs, ascending=False),
        "dropped_low_corr": dropped_low_corr,
        "dropped_collinear": dropped_collinear,
    }

In [ ]:
# 3. Feature selection -- fit on X_train/y_train only (as before)
result = select_features_spearman(X_train, y_train)
selected = result["selected"]

X_train_sel = X_train[selected]
X_test_sel = X_test[selected]

print(f"chunk: {len(train_raw)} train / {len(test_raw)} test days")
print(f"{len(selected)} of {X_train.shape[1]} features kept")

In [ ]:

def plot_selected_features(result: dict, top_n: int = None):
    """
    Bar chart of Spearman rho (vs target) for the features `select_features_spearman`
    kept. Sign is preserved -- positive rho (feature rises with price) in one color,
    negative in another.
    """
    selected = result["selected"]
    corr = result["target_corr"][selected]

    if top_n is not None:
        corr = corr.reindex(corr.abs().sort_values(ascending=False).index[:top_n])

    corr = corr.sort_values()  # ascending, so strongest bars end up at the top when plotted

    colors = ["#1D9E75" if v > 0 else "#D85A30" for v in corr]

    fig, ax = plt.subplots(figsize=(8, max(4, len(corr) * 0.3)))
    ax.barh(corr.index, corr.values, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Spearman rho with price_mean_d1")
    ax.set_title(f"Selected features ({len(selected)} of {len(result['target_corr'])} total)")
    plt.tight_layout()
    plt.show()


plot_selected_features(result)                # all selected features
# plot_selected_features(result, top_n=25)    # just the strongest 25, if the full list is too tall

# SAMIMA MODEL

In [ ]:
full_idx = pd.date_range(y_train.index.min(), y_train.index.max(), freq='D')
n_missing = len(full_idx) - len(y_train)
print(f"Missing days in train: {n_missing}")

y_train = y_train.reindex(full_idx)
y_train = y_train.interpolate(limit=3)   # only bridge short gaps, don't paper over long ones
y_train.index.freq = 'D'

In [ ]:
adf_stat, adf_p, *_ = adfuller(y_train.dropna())
print(f"ADF stat: {adf_stat:.3f}, p-value: {adf_p:.4f}")  # p < 0.05 -> stationary

decomp = seasonal_decompose(y_train.dropna(), period=7, model='additive')
decomp.plot()

In [ ]:
from statsmodels.tsa.stattools import kpss

kpss_stat, kpss_p, *_ = kpss(y_train.dropna(), regression='c', nlags='auto')
print(f"KPSS stat: {kpss_stat:.3f}, p-value: {kpss_p:.4f}")
# p > 0.05 -> fail to reject stationarity -> agrees with ADF
# p < 0.05 -> reject stationarity -> conflicts with ADF, worth more digging

In [ ]:
def walk_forward_sarima(
    y: pd.Series,
    n_folds: int = 5,
    horizon: int = 14,
    min_train_size: int = 365,
    m: int = 7,
    auto_arima_kwargs: dict | None = None,
) -> pd.DataFrame:
    """
    Expanding-window walk-forward validation para auto_arima + SARIMAX.
 
    En cada fold:
      - train = y[:cutoff]  (crece en cada fold, nunca se reduce)
      - test  = y[cutoff : cutoff + horizon]
      - se corre auto_arima sobre train para elegir el orden de ese fold
      - se ajusta SARIMAX con ese orden y se predice `horizon` pasos
      - se guarda el orden elegido y las métricas del fold
 
    Parameters
    ----------
    y : Serie objetivo (ej. tu y_train ya interpolado/reindexado a 'D')
    n_folds : número de ventanas de validación
    horizon : pasos a predecir en cada fold (ej. 14 días)
    min_train_size : tamaño mínimo del primer train (evita folds con
        muy pocos datos, donde auto_arima es poco fiable)
    m : periodo estacional (7 = semanal, como ya usas)
    auto_arima_kwargs : kwargs extra para pm.auto_arima (por defecto
        replica lo que ya tienes en la celda 18)
 
    Returns
    -------
    DataFrame con una fila por fold: fold, cutoff_date, order,
    seasonal_order, mae, rmse, smape
    """
    if auto_arima_kwargs is None:
        auto_arima_kwargs = dict(
            seasonal=True, d=None, D=None, stepwise=True,
            error_action="ignore", suppress_warnings=True, trace=False,
        )
 
    n = len(y)
    # tamaño de cada paso: reparte el espacio disponible tras min_train_size
    # entre los n_folds, dejando sitio para el horizonte de cada fold
    usable = n - min_train_size - horizon
    if usable <= 0:
        raise ValueError(
            f"Serie demasiado corta para {n_folds} folds: "
            f"n={n}, min_train_size={min_train_size}, horizon={horizon}"
        )
    step = max(1, usable // n_folds)
 
    rows = []
    for fold in range(n_folds):
        cutoff = min_train_size + fold * step
        train = y.iloc[:cutoff].dropna()
        test = y.iloc[cutoff: cutoff + horizon].dropna()
        if len(test) < horizon:
            break  # nos quedamos sin datos suficientes para este fold
 
        auto = pm.auto_arima(train, m=m, **auto_arima_kwargs)
        order, seasonal_order = auto.order, auto.seasonal_order
 
        fit = SARIMAX(
            train, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False,
        ).fit(disp=False)
        pred = fit.forecast(steps=len(test))
        pred.index = test.index
 
        mae = mean_absolute_error(test, pred)
        rmse = np.sqrt(mean_squared_error(test, pred))
        smape = np.mean(2 * np.abs(test - pred) / (np.abs(test) + np.abs(pred))) * 100
 
        rows.append({
            "fold": fold,
            "cutoff_date": train.index[-1],
            "train_size": len(train),
            "order": order,
            "seasonal_order": seasonal_order,
            "mae": mae,
            "rmse": rmse,
            "smape": smape,
        })
        print(f"fold {fold}: cutoff={train.index[-1].date()} "
              f"order={order}{seasonal_order} MAE={mae:.2f} RMSE={rmse:.2f}")
 
    results = pd.DataFrame(rows)
 
    # Resumen de estabilidad: ¿cuántos órdenes distintos salieron?
    order_counts = Counter(
        (r["order"], r["seasonal_order"]) for r in rows
    )
    print("\nEstabilidad del orden elegido entre folds:")
    for (order, sorder), count in order_counts.most_common():
        print(f"  {order}{sorder}: {count}/{len(rows)} folds")
 
    return results

In [ ]:
walk_forward_sarima(y_train, n_folds=5, horizon=14, min_train_size=365, m=7)

In [ ]:


auto = pm.auto_arima(
    y_train, seasonal=True, m=7,
    d=None, D=None,           # let it search both
    stepwise=True, trace=True,
    error_action='ignore', suppress_warnings=True
)
print(auto.order, auto.seasonal_order)



In [ ]:
order = auto.order
seasonal_order = auto.seasonal_order
print(order)
print(seasonal_order)

In [ ]:

sarima_fit = SARIMAX(
    y_train, order= order, seasonal_order=seasonal_order,
    enforce_stationarity=False, enforce_invertibility=False
).fit(disp=False)
print(sarima_fit.summary())

In [ ]:
predictions = []
current_fit = sarima_fit
for t in range(len(y_test)):
    pred = current_fit.forecast(steps=1)
    predictions.append(pred.iloc[0])
    current_fit = current_fit.append([y_test.iloc[t]], refit=False)  # updates state, no refit cost

predictions = pd.Series(predictions, index=y_test.index)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae  = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
smape = np.mean(2 * np.abs(y_test - predictions) / (np.abs(y_test) + np.abs(predictions))) * 100
print(f"MAE: {mae:.2f} €/MWh  RMSE: {rmse:.2f} €/MWh  SMAPE: {smape:.2f}%")

In [ ]:
# naive persistence: tomorrow's price = today's price
naive_pred = y_test.shift(1).bfill()

# seasonal naive: tomorrow's price = same day last week
seasonal_naive_pred = y_test.shift(7).bfill()

for name, pred in [("Naive (t-1)", naive_pred), ("Seasonal naive (t-7)", seasonal_naive_pred)]:
    mae_b = mean_absolute_error(y_test, pred)
    rmse_b = np.sqrt(mean_squared_error(y_test, pred))
    print(f"{name}: MAE={mae_b:.2f}  RMSE={rmse_b:.2f}")